# 청크 비교 — 표·이미지 추출 미적용 vs 적용

| | 파일 | 구성 |
|---|---|---|
| 기존 | `chunks_800_120.jsonl` | 본문만. 표는 구분자 없이 뭉개져 있고 이미지 내용은 없음 |
| 신규 | `chunks_extraction_800_120.jsonl` | 본문(표 제외) + 표 청크 + 이미지 청크 |

기존 파일은 그대로 두고 새 파일을 따로 만들었다. 팀원들이 기존 청크로 실험 중이기 때문이다.

검색은 두 세트를 **같은 임베딩 모델로 각각 색인**해 비교했다(`text-embedding-3-small`).
생성은 `gpt-5-nano`가 비결정적이라 문항마다 3회 반복해 평균을 냈다.


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import HTML, display

# 노트북을 notebook/ 안에서 실행해도 프로젝트 루트를 기준으로 동작하게 맞춥니다.
if Path.cwd().name == "notebook":
    os.chdir("..")

CONFIG = yaml.safe_load(open("config/default.yaml", encoding="utf-8"))
RESULT_PATH = Path.home() / "vision_cmp" / "chunk_comparison.json"


def load_jsonl(path):
    with Path(path).open(encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def source_of(chunk):
    return chunk["metadata"].get("chunk_source", "body")


before = load_jsonl(CONFIG["paths"]["chunks"])
after = load_jsonl(CONFIG["paths"]["chunks_with_extraction"])
results = pd.DataFrame(json.loads(RESULT_PATH.read_text(encoding="utf-8")))
print(f"기존 {len(before):,}개  ->  신규 {len(after):,}개")

기존 10,222개  ->  신규 17,382개


## 1. 청크 구성이 어떻게 바뀌었나

In [2]:
rows = []
for label, chunks in (("기존", before), ("신규", after)):
    counts = pd.Series([source_of(chunk) for chunk in chunks]).value_counts()
    rows.append({
        "구분": label,
        "본문": counts.get("body", 0),
        "표": counts.get("table", 0),
        "이미지": counts.get("image", 0),
        "합계": len(chunks),
        "문서": len({chunk["doc_id"] for chunk in chunks}),
    })
display(pd.DataFrame(rows).set_index("구분"))

print(f"청크 수 {len(after) / len(before):.1f}배")
print("본문 청크가 줄어든 것은 표 내용이 본문에서 빠져 표 청크로 옮겨갔기 때문이다.")
print("장·절 제목은 표로 만들어져 있어도 본문에 남겼다. 빼면 목차가 어디에도 남지 않는다.")

,본문,표,이미지,합계,문서
구분,,,,,
기존,10222,0,0,10222,100
신규,4197,12892,293,17382,100


청크 수 1.7배
본문 청크가 줄어든 것은 표 내용이 본문에서 빠져 표 청크로 옮겨갔기 때문이다.
장·절 제목은 표로 만들어져 있어도 본문에 남겼다. 빼면 목차가 어디에도 남지 않는다.


In [3]:
display(pd.DataFrame({
    "기존 본문": pd.Series([len(c["text"]) for c in before]),
    "신규 본문": pd.Series([len(c["text"]) for c in after if source_of(c) == "body"]),
    "신규 표": pd.Series([len(c["text"]) for c in after if source_of(c) == "table"]),
    "신규 이미지": pd.Series([len(c["text"]) for c in after if source_of(c) == "image"]),
}).describe().round(0))

,기존 본문,신규 본문,신규 표,신규 이미지
count,10222.0,4197.0,12892.0,293.0
mean,797.0,792.0,439.0,420.0
std,39.0,58.0,270.0,295.0
min,125.0,128.0,20.0,20.0
25%,800.0,800.0,232.0,93.0
50%,800.0,800.0,397.0,457.0
75%,800.0,800.0,663.0,731.0
max,800.0,800.0,3937.0,799.0


## 2. 표와 이미지가 실제로 담겼는가

숫자만으로는 내용이 제대로 들어갔는지 알 수 없다. 실제 청크를 꺼내 본다.

In [4]:
def escape(text):
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def show(title, subtitle, body):
    display(HTML(f"""
    <div style="margin:14px 0;">
      <div style="font-weight:bold;">{escape(title)}</div>
      <div style="font-size:12px; margin-bottom:6px;">{escape(subtitle)}</div>
      <pre style="white-space:pre; overflow:auto; max-height:340px; font-size:12px;
                  margin:0; padding:10px; border:1px solid #ccc; border-radius:6px;">{escape(body)}</pre>
    </div>"""))


tables = [c for c in after if source_of(c) == "table"]
bodies = [c for c in after if source_of(c) == "body"]
pictures = [c for c in after if source_of(c) == "image"]

print("표 청크에 격자가 살아 있는가")
print(f"  | 포함      {sum(1 for c in tables if '|' in c['text']):,} / {len(tables):,}")
print(f"  줄바꿈 포함 {sum(1 for c in tables if chr(10) in c['text']):,} / {len(tables):,}")
print()
print("본문 청크에 표가 남아 있는가")
print(f"  | 포함 {sum(1 for c in bodies if '|' in c['text']):,} / {len(bodies):,}")

표 청크에 격자가 살아 있는가
  | 포함      12,892 / 12,892
  줄바꿈 포함 12,892 / 12,892

본문 청크에 표가 남아 있는가
  | 포함 1 / 4,197


In [5]:
show(
    f"표 청크 — {tables[0]['chunk_id']}",
    f"{tables[0]['metadata'].get('extraction_method')} / {len(tables[0]['text'])}자",
    tables[0]["text"][:900],
)

picture = next(c for c in pictures if c["doc_id"] == "doc_011")
show(
    f"이미지 청크 — {picture['chunk_id']}",
    f"유형 {picture['metadata'].get('image_type')} / {len(picture['text'])}자",
    picture["text"][:900],
)

이미지 청크는 첫 줄에 유형과 요약을 세웠다.
뒤따르는 OCR 결과는 맥락 없는 낱말 나열이라, 그것만으로는 문장으로 들어오는 질문과 임베딩 거리가 멀다.
이 한 줄을 앞으로 옮긴 뒤 Q13의 정답 키워드가 1.0/3에서 2.3/3으로 올랐다.

### 같은 표가 기존에는 어떻게 들어 있었나

In [6]:
probe = None
for chunk in tables:
    lines = chunk["text"].splitlines()
    if len(lines) < 2:
        continue
    cells = [cell.strip() for cell in lines[1].strip("|").split("|") if len(cell.strip()) >= 8]
    if cells:
        probe = (chunk, cells[0])
        break

table_chunk, keyword = probe
matched = [c for c in before if keyword in c["text"] and c["doc_id"] == table_chunk["doc_id"]]
print(f"찾는 문구: {keyword!r}\n")
show("신규 · 표 청크", table_chunk["chunk_id"], "\n".join(table_chunk["text"].splitlines()[:8]))
if matched:
    position = matched[0]["text"].find(keyword)
    show("기존 · 본문 청크", matched[0]["chunk_id"], matched[0]["text"][max(0, position - 200):position + 300])

찾는 문구: '1. 사업수행 능력 2. 유사분야 사업 실적'



## 3. 공통질문 13개 결과

같은 질문을 두 세트에 각각 던졌다. 검색은 한 번(결정적), 생성은 3회 반복해 평균을 냈다.

In [7]:
summary = results.groupby(["question_id", "set"]).agg(
    유형=("type", "first"),
    키워드=("keyword_hit", "mean"),
    총=("keyword_total", "first"),
    정답문서=("doc_hit", "first"),
    표=("table_chunks", "first"),
    이미지=("image_chunks", "first"),
).reset_index()

pivot = summary.pivot(index="question_id", columns="set")
table = pd.DataFrame({
    "유형": pivot[("유형", "기존")],
    "키워드 기존": pivot[("키워드", "기존")].round(1).astype(str) + "/" + pivot[("총", "기존")].astype(str),
    "키워드 신규": pivot[("키워드", "신규")].round(1).astype(str) + "/" + pivot[("총", "신규")].astype(str),
    "문서 기존": pivot[("정답문서", "기존")],
    "문서 신규": pivot[("정답문서", "신규")],
    "표": pivot[("표", "신규")],
    "이미지": pivot[("이미지", "신규")],
})
display(table)

results["rate"] = results["keyword_hit"] / results["keyword_total"]
print(f"키워드 적중률   기존 {results[results['set'] == '기존']['rate'].mean() * 100:.1f}%"
      f"  ->  신규 {results[results['set'] == '신규']['rate'].mean() * 100:.1f}%")
print(f"정답문서 평균   기존 {summary[summary['set'] == '기존']['정답문서'].mean():.2f}"
      f"   ->  신규 {summary[summary['set'] == '신규']['정답문서'].mean():.2f}")

,유형,키워드 기존,키워드 신규,문서 기존,문서 신규,표,이미지
question_id,,,,,,,
Q01,single,3.0/3,3.0/3,2,3,2,0
Q02,numeric,2.0/2,2.0/2,2,4,2,0
Q03,single,3.0/3,3.0/3,3,4,1,0
Q04,single,3.0/3,3.0/3,5,4,4,0
Q05,table,0.0/6,6.0/6,0,1,2,0
Q06,table,3.0/3,3.0/3,2,4,3,1
Q07,multi_document,2.0/4,2.0/4,2,3,2,0
Q08,unsupported,0.7/2,0.7/2,0,0,2,0
Q09,follow_up,2.0/2,0.0/2,5,4,4,1


키워드 적중률   기존 59.8%  ->  신규 65.0%
정답문서 평균   기존 2.69   ->  신규 3.23


### 질문별 답변 비교

In [8]:
for question_id in sorted(results["question_id"].unique()):
    part = results[results["question_id"] == question_id]
    item = part.iloc[0]
    panels = ""
    for label in ("기존", "신규"):
        one = part[(part["set"] == label) & (part["trial"] == 1)].iloc[0]
        mean_hit = part[part["set"] == label]["keyword_hit"].mean()
        panels += f"""
        <div style="flex:1; min-width:0; border:1px solid #999; border-radius:6px; padding:10px;">
          <div style="font-weight:bold;">{label}</div>
          <div style="font-size:12px; margin-bottom:6px;">
            키워드 {mean_hit:.1f}/{one['keyword_total']} (3회 평균) ·
            정답문서 {one['doc_hit']} · 표 {one['table_chunks']} · 이미지 {one['image_chunks']}
          </div>
          <pre style="white-space:pre-wrap; font-size:12px; margin:0;">{escape(one['answer'])}</pre>
        </div>"""
    display(HTML(f"""
    <div style="margin:20px 0;">
      <div style="font-weight:bold;">{question_id} · {item['type']}</div>
      <div style="font-size:13px; margin:4px 0;">{escape(item['question'])}</div>
      <div style="font-size:12px; margin-bottom:8px;">정답 키워드: {escape(', '.join(item['keywords']))}</div>
      <div style="display:flex; gap:12px; align-items:flex-start;">{panels}</div>
    </div>"""))

## 정리

**좋아진 것**

- 키워드 적중률 59.8% → 65.0%, 정답문서 평균 2.69 → 3.23
- Q05(표 문항)가 0.0/6에서 6.0/6으로 올랐다. 기존에는 정답 문서를 하나도 찾지 못하던 문항이다
- 정답 문서 검색이 7문항에서 개선됐다 (Q01, Q02, Q03, Q05, Q06, Q07, Q11)

**나빠진 것**

- Q09(후속 질문)가 2.0/2에서 0.0/2로 떨어졌다. 3회 모두 실패해 노이즈가 아니다
- 정답이 담긴 본문 청크가 표 청크에 밀려 상위 5개 안에 들어오지 못했다

**원인과 인계 사항**

표 청크가 12,892개로 본문(4,197개)의 약 3배다. `top_k: 5`에서는 상위 결과를 표가 잠식한다.
13문항 중 10문항에서 표 청크가 2개 이상 검색됐다.

`top_k` 상향 또는 청크 종류별 할당은 Retrieval 설정 범위이므로 담당자에게 전달한다.
